SAMPLE RATE: 8000 Hz  
Window size: 40 ms  
Shift size: 10 ms  
Frames received each 10 ms

In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
from spicy import signal
import pickle
import warnings
import gzip
import scipy.io
#from scipy.io import wavfile
#from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

sys.path.append('/home/adelval/BTS/TFM/afterburner8k/src/net')
sys.path.append('/home/adelval/BTS/TFM/afterburner8k/src/train')
sys.path.append('/home/adelval/BTS/TFM/afterburner8k/src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [2]:
x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
print('  x_test: %s' % (x_test))

  x_test: ['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [3]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [4]:
audio, fs = read_audio(x_test[0])
print(audio)
print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

frame_size = 0.01 # 10 ms
frame_samples = int(frame_size * fs)
print(f'Un frame tiene una duración de {frame_samples} samples')

[  0  -1  -1 ... -30 -34 -30]
La duración del audio es 4.655 segundos y 74480 muestras
Un frame tiene una duración de 160 samples


In [5]:
# FFT 
def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)
    
def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x
    
def windowing2(x, fs=16000, Ns=0.040, Ms=0.010):
    N = int(Ns * fs)
    M = int(Ms * fs)
    n = (len(x) + M - 1) // M    
    T = (n - 1) * M + N 
    xa = x.copy()
    if T > len(x):
        xa.resize(T, refcheck=False)
    m = np.arange(0, n * M, M)
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1)
    return xa[ind.astype(int).T].astype(np.float32)

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def window_fft(data, fs, w, nfft):
    
    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]

    x = offset(data)

    # Emphasis to increase the amplitude of high freq
    x = preemphasis(x)

    XX = [] # Aqui almacenare hasta 20 ventanas para dar contexto

    X = hamming(x)
    print("El frame enventanado tiene dimensión: ",X.shape)
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    print("Vector con Power Spectral Density del frame: \n", X[:10])

    return X    # Returns the PSD of the n windows procesed

# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   

In [ ]:
fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
gmin = 0.0562

N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]

In [ ]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b

def window_fb_mfcc(data, fs, B, w, nfft):

    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
    fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
    #print(f'fb es {len(fb[0])}')
    dct = [ f_base_dct(Bi) for Bi in B] 
    #print(f'dct es {len(dct[0])}')


    x = offset(data)
    x = preemphasis(x)
    XX = []
    for i,w in enumerate(w):
        X = hamming(x)
        
        Xfft = fft(X, nfft[i])
        Xb = np.log(Xfft.dot( fb[i] ) + 1)
        Xc = Xb.dot(dct[i])                                
        
        X = np.concatenate( [Xb, Xc], 0 )
        
        X = np.asarray(X, dtype=np.float32)
        XX.append(X)
    XX = np.concatenate(XX, 0)
    
    #print("El tamaño de x08k2 es: ",XX.shape)
    print("Vector con FilterBank MFCC del frame: \n", XX[:10])
    
    return XX

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc):
    # Normalization of fbmfcc
    file = '/home/adelval/BTS/TFM/test/data/model/fe1_norm1.pkl'  # de donde salen??
    x = frame_fbmfcc
    mu, std = read_pkl(file)
    x -= mu
    x /= std + 1e-6
    print("Vector con FB MFCC normalizado del frame: \n", x[:10])

    return x

In [8]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj('/home/adelval/BTS/TFM/afterburner8k/data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append('/home/adelval/BTS/TFM/afterburner8k/src/net')

# Cargo la original porque la por bloques da fallo de dimensiones
from net_snr_original import Net_snr
net_snr = Net_snr(input_dim, output_dim, cuda=True, single_gpu=True)
net_snr.load('/home/adelval/BTS/TFM/afterburner8k/data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:
    nb_params: 29.99M
    cuda: True
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    restoring epoch: 500
    restoring opt: adama, lr: 0.000019
    opt: adama, 1.94812e-05, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 1.9481214405699424e-05
    weight_decay: 0
)
    reading /home/adelval/BTS/TFM/afterburner8k/data/model/theta_last


500

In [ ]:
audio, fs = read_audio(x_test[0])

fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
gmin = 0.0562


print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

frame_size = 0.01 # 10 ms
frame_samples = int(frame_size * fs)
print(f'Un frame tiene una duración de {frame_samples} samples')


# Simulates Incoming Buffer 
shift_size = 0.01 # 10 ms
shift_samples = int(shift_size * fs)
window_size = 0.04 # 40ms
window_samples = int(window_size * fs)
print(f'La ventana tiene una duración de {window_samples} samples')

buffer_frame = []
for n_frame in range(4):
    # First buffer need to have 4 frames
    if(len(buffer_frames) < window_samples):
        buffer_frame = np.concatenate([buffer_frame,audio[n_frame*shift_samples:n_frame*shift_samples+frame_samples]])
    else:
        buffer_frame[:(window_samples-shift_samples)] = buffer_frame[shift_samples:]
        buffer_frame[(window_samples-shift_samples):] = audio[n_frame*shift_samples:n_frame*shift_samples+frame_samples]
    """
    print("Iteration number ", n_frame)
    print("1",buffer_frame[:10])
    print("2",buffer_frame[160:170])
    print("3",buffer_frame[320:330])
    print("4",buffer_frame[480:490])
    """
    ## PROCESADO
    if(len(buffer_frame) >= window_samples):
        #print(buffer_frame.shape)
        frame_psd = frame_fft(buffer_frame, fs, w, nfft)
        frame_psd_log = log_scale(frame_psd)
        frame_fbmfcc = frame_fb_mfcc(buffer_frame, fs, B, w, nfft)
        frame_fbmfcc_norm = norm_fb_frame(frame_fbmfcc)
        frame_concat = np.concatenate( (frame_psd_log,frame_fbmfcc_norm), 0 )
        snr_frame_mask = net_eval(frame_concat)
        snr_frame_mask = snr_frame_mask[:, np.newaxis]
        print(f'La snr cargada es de dimensiones {snr_frame_mask.shape}')
        print(f'La máscara del primer fragmento es {snr_frame_mask}')
        
        x = np.array(buffer_frame, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
        print(f'El frame sin enventanado resulta {x[:10]}')
        xenh, filt = noiseReduction(x, snr_frame_mask, fs, window_samples, shift_samples, nfft[0], gmin)
        print(f'Las dimensiones del filtro son {filt.shape}')
        print(f'El frame mejorado es {xenh[:10]}')
        #yenh[]
    # Simulates Outgoing 

    